# 🎧 IELTS Dictation — Auto Timestamp Generator (Kaggle)

**Full-auto pipeline:** Upload audio → Run All → Download JSON

### One-time setup:
1. **GPU**: Settings → Accelerator → **GPU T4 x2**
2. **Internet**: Settings → Internet → **On**
3. Upload audio: **+ Add Input** → Upload MP3

## ⚙️ CONFIG — Edit this cell, then Run All

In [ ]:
# ══════════════════════════════════════════════════════════
# EDIT THESE VALUES THEN HIT: Run → Run All
# ══════════════════════════════════════════════════════════

AUDIO_FILE = "/kaggle/input/YOUR_DATASET/your_audio.mp3"

EXERCISE_TITLE = "Cam20 - Test 1 - Part 1"
EXERCISE_DESCRIPTION = "Restaurant Recommendations"
EXERCISE_LEVEL = "B2"              # A1, A2, B1, B2, C1, C2
EXERCISE_CATEGORY = "Cambridge 20"

# ── Optional filters ─────────────────────────────────────
SKIP_BEFORE_SECONDS = 0    # bỏ audio trước mốc này (giây)
SKIP_FIRST_N = 0           # bỏ N câu đầu tiên
MIN_WORDS_TO_MERGE = 3     # merge câu ngắn hơn N từ vào câu trước
WHISPER_MODEL = "large-v3-turbo"

# ══════════════════════════════════════════════════════════
# NOTHING TO EDIT BELOW — everything runs automatically
# ══════════════════════════════════════════════════════════

## 🚀 Auto Pipeline

In [ ]:
# ─── Step 1: Install ─────────────────────────────────────
!pip install -q openai-whisper
import os, json, re, whisper, torch

assert os.path.exists(AUDIO_FILE), f"❌ File not found: {AUDIO_FILE}\nRun this to find files: os.walk('/kaggle/input')"
print(f"✅ Audio: {AUDIO_FILE}")

In [ ]:
# ─── Step 2: Transcribe ──────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️ {device.upper()} | Model: {WHISPER_MODEL}")

model = whisper.load_model(WHISPER_MODEL, device=device)
result = model.transcribe(AUDIO_FILE, language="en", word_timestamps=True, verbose=False)
print(f"✅ Transcribed! {len(result['segments'])} raw segments")

In [ ]:
# ─── Step 3: Clean — all filters in one shot ─────────────

# 3a. Extract sentences
sentences = []
for seg in result["segments"]:
    text = seg["text"].strip()
    if text:
        sentences.append({
            "index": len(sentences),
            "text": text,
            "startTime": round(seg["start"], 2),
            "endTime": round(seg["end"], 2)
        })
total_raw = len(sentences)

# 3b. Remove Whisper HALLUCINATIONS
# - Duration < 0.5s = too short to be real speech
# - Contains non-Latin characters (Chinese, Japanese, Korean, etc.)
# - Extremely repetitive text from silence
NON_LATIN = re.compile(r'[\u4e00-\u9fff\u3000-\u303f\uff00-\uffef\u3040-\u309f\u30a0-\u30ff\uac00-\ud7af]')

hallucination_count = 0
clean = []
for s in sentences:
    duration = s["endTime"] - s["startTime"]
    has_non_latin = bool(NON_LATIN.search(s["text"]))
    is_too_short = duration < 0.5 and len(s["text"].split()) > 3
    
    if has_non_latin or is_too_short:
        hallucination_count += 1
        print(f"  🗑️ Hallucination: [{s['startTime']:.1f}s-{s['endTime']:.1f}s] \"{s['text'][:50]}\"")
    else:
        clean.append(s)
sentences = clean

# 3c. Skip intro (by time or count)
skipped_intro = 0
if SKIP_FIRST_N > 0:
    sentences = sentences[SKIP_FIRST_N:]
    skipped_intro += SKIP_FIRST_N
if SKIP_BEFORE_SECONDS > 0:
    before_skip = len(sentences)
    sentences = [s for s in sentences if s["startTime"] >= SKIP_BEFORE_SECONDS]
    skipped_intro += before_skip - len(sentences)

# 3d. Auto-remove narrator instructions
INSTRUCTION_KEYWORDS = [
    "questions", "look at", "you will hear", "listen carefully",
    "now turn to", "that is the end", "before you hear",
    "you have some time", "now listen", "section",
    "part one", "part two", "part three", "part four",
    "example", "answer the questions", "read the questions",
    "first you have", "complete the", "choose the correct",
    "write no more than", "label the",
    "has been written", "now we shall begin", "we shall begin",
    "the answer is", "so the answer", "you can see",
    "in the space", "on the form", "on your answer sheet",
    "end of section", "end of part",
]

removed_instructions = []
clean = []
for s in sentences:
    text_lower = s["text"].lower()
    matched = [kw for kw in INSTRUCTION_KEYWORDS if kw in text_lower]
    if matched:
        removed_instructions.append((s["text"][:60], matched))
    else:
        clean.append(s)
sentences = clean

# 3e. Detect and remove DUPLICATE intro (example section)
# IELTS often plays the example once, then replays the conversation.
# Detect by finding repeated text in the first ~20 sentences.
dupe_removed = 0
if len(sentences) > 10:
    # Check if sentences 0-4 text matches sentences somewhere in 5-15
    first_text = sentences[0]["text"].lower().strip()[:40] if sentences else ""
    dupe_start = -1
    for i in range(3, min(15, len(sentences))):
        if sentences[i]["text"].lower().strip()[:40] == first_text:
            dupe_start = i
            break
    
    if dupe_start > 0:
        dupe_removed = dupe_start
        sentences = sentences[dupe_start:]
        print(f"  🔁 Removed {dupe_removed} duplicated example sentences (intro replay detected)")

# 3f. Merge short segments
merged = []
for s in sentences:
    if merged and len(s["text"].split()) < MIN_WORDS_TO_MERGE:
        merged[-1]["text"] += " " + s["text"]
        merged[-1]["endTime"] = s["endTime"]
    else:
        merged.append(dict(s))
sentences = merged

# 3g. Re-index
for i, s in enumerate(sentences):
    s["index"] = i

# ─── Report ──────────────────────────────────────────────
print(f"\n📊 Pipeline Results:")
print(f"   Raw segments:       {total_raw}")
print(f"   Hallucinations:     {hallucination_count} removed")
print(f"   Intro skipped:      {skipped_intro}")
print(f"   Duplicate example:  {dupe_removed} removed")
print(f"   Instructions:       {len(removed_instructions)} removed")
print(f"   Final:              {len(sentences)} sentences ✅")

if removed_instructions:
    print(f"\n🗑️ Removed instructions:")
    for text, kws in removed_instructions:
        print(f"   ✂️ \"{text}\"  [{', '.join(kws[:3])}]")

In [ ]:
# ─── Step 4: Export JSON ─────────────────────────────────
output = {
    "title": EXERCISE_TITLE,
    "description": EXERCISE_DESCRIPTION,
    "level": EXERCISE_LEVEL,
    "category": EXERCISE_CATEGORY,
    "audioFileName": os.path.basename(AUDIO_FILE),
    "totalSentences": len(sentences),
    "sentences": sentences
}

out_name = os.path.basename(AUDIO_FILE).rsplit('.', 1)[0] + "_dictation.json"
out_path = f"/kaggle/working/{out_name}"

with open(out_path, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print(f"\n✅ DONE! Saved: {out_path}")
print(f"📊 {len(sentences)} sentences")
print(f"📥 Download from Output panel (right side)")

## 👀 Preview (optional)

In [ ]:
print(f"{'#':<4} {'Start':>6} {'End':>6}  Text")
print("-" * 80)
for s in sentences:
    print(f"{s['index']:<4} {s['startTime']:>6.1f} {s['endTime']:>6.1f}  {s['text'][:60]}")

## 🔧 Manual Fix (nếu cần sửa)

Chỉ dùng nếu Whisper nghe sai từ nào đó:

In [ ]:
# Uncomment để sửa:
# sentences[0]["text"] = "Corrected text here"
# sentences[5]["text"] = "Another correction"
#
# Sau đó chạy lại cell Export (Step 4) ở trên

---
## 📋 Import vào database

```bash
curl -X POST http://localhost:3003/api/dictation \
  -H 'Content-Type: application/json' \
  -d @your_file_dictation.json
```